# SQL Cross-Validation
### Global Electronics Retailer

This short notebook proves the pandas-based analysis in modules 1-4 and the pure-SQL queries in `sql/queries.sql` agree exactly. The same 5 CSVs are loaded into a SQLite database (`data/global_electronics.db`, built by `src/build_database.py`) and re-queried in SQL, independent of the pandas pipeline in `src/data_prep.py`.

Running both approaches and confirming they match is a basic but important sanity check before trusting either one.


In [1]:
import sys
sys.path.insert(0, "../src")
import sqlite3
import pandas as pd
from data_prep import build_sales_fact_table

pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

# Pandas pipeline (as used in modules 1-4)
df = build_sales_fact_table()

# SQL pipeline (independent path, same raw CSVs)
conn = sqlite3.connect("../data/global_electronics.db")
conn.executescript(open("../sql/queries.sql").read().split("-- 1.1")[0])

### Check 1: Total Revenue & Profit

In [2]:
pandas_total = df["Revenue_USD"].sum()
sql_total = conn.execute("SELECT SUM(revenue_usd) FROM sales_fact").fetchone()[0]
print(f"pandas: ${pandas_total:,.2f}")
print(f"SQL:    ${sql_total:,.2f}")
assert round(pandas_total, 2) == round(sql_total, 2), "Mismatch!"
print("MATCH")

pandas: $55,755,479.59
SQL:    $55,755,479.59
MATCH


### Check 2: Year-over-Year Growth

In [3]:
full_years = df[df["Order_Year"].between(2016, 2020)]
pandas_yoy = full_years.groupby("Order_Year")["Revenue_USD"].sum().pct_change().dropna() * 100
print("pandas:\n", pandas_yoy.round(1))

sql_yoy = pd.read_sql('''
    WITH annual AS (
        SELECT strftime('%Y', order_date) AS yr, SUM(revenue_usd) AS revenue
        FROM sales_fact WHERE strftime('%Y', order_date) BETWEEN '2016' AND '2020' GROUP BY yr
    )
    SELECT yr, ROUND(100.0*(revenue-LAG(revenue) OVER (ORDER BY yr))/LAG(revenue) OVER (ORDER BY yr),1) AS yoy
    FROM annual ORDER BY yr
''', conn)
print("\nSQL:\n", sql_yoy)

pandas:
 Order_Year
2017     6.80
2018    72.30
2019    42.80
2020   -49.10
Name: Revenue_USD, dtype: float64



SQL:
      yr    yoy
0  2016    NaN
1  2017   6.80
2  2018  72.30
3  2019  42.80
4  2020 -49.10


### Check 3: Top-10% Customer Revenue Concentration

In [4]:
rfm = df.groupby("CustomerKey")["Revenue_USD"].sum().sort_values(ascending=False)
top10_pandas = rfm.head(int(len(rfm)*0.1)).sum() / rfm.sum() * 100

top10_sql = conn.execute('''
    WITH rfm AS (SELECT CustomerKey, SUM(revenue_usd) AS monetary FROM sales_fact GROUP BY CustomerKey),
    ranked AS (SELECT *, PERCENT_RANK() OVER (ORDER BY monetary DESC) AS pct_rank FROM rfm)
    SELECT ROUND(100.0*SUM(CASE WHEN pct_rank<=0.10 THEN monetary ELSE 0 END)/SUM(monetary),1) FROM ranked
''').fetchone()[0]

print(f"pandas: {top10_pandas:.1f}%")
print(f"SQL:    {top10_sql:.1f}%")

pandas: 36.0%
SQL:    36.0%


### Check 4: Zero-Revenue Stores (Data Quality Flag)

In [5]:
sql_zero_stores = pd.read_sql('''
    SELECT st.StoreKey, st.Country FROM stores st
    LEFT JOIN sales_fact f ON f.StoreKey = st.StoreKey
    WHERE st.StoreKey != 0 GROUP BY st.StoreKey
    HAVING COALESCE(SUM(f.revenue_usd), 0) = 0
''', conn)
print(f"{len(sql_zero_stores)} stores with zero recorded sales (matches the 9 found in Module 2)")
sql_zero_stores

9 stores with zero recorded sales (matches the 9 found in Module 2)


,StoreKey,Country
0,3,Australia
1,7,Canada
2,11,Canada
3,25,Germany
4,35,Netherlands
5,46,United States
6,52,United States
7,58,United States
8,60,United States


**All checks pass** — the pandas notebooks (01-04) and the SQL layer (`sql/queries.sql`) are two independent, mutually-verified paths to the same numbers. Either can be used to reproduce or audit this analysis.